# HW14: Vector Embeddings, FAISS Index, Retrieval Quality, and mini-RAG

Тема: эмбеддинги, индекс FAISS, оценка качества retrieval, обновление базы знаний и mini-RAG.

## 1. Импорты, seed и среда

In [1]:
%pip install faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 87.6 MB/s eta 0:00:00


In [2]:
import os
import sys
import random
import numpy as np
import pandas as pd
import json
from typing import List, Dict, Tuple
import warnings
warnings.filterwarnings('ignore')

# Vectorization and search
import faiss
from sklearn.metrics.pairwise import cosine_similarity

# Embeddings
try:
    from sentence_transformers import SentenceTransformer
except ImportError:
    print("Installing sentence-transformers...")
    import subprocess
    subprocess.check_call([sys.executable, "-m", "pip", "install", "sentence-transformers", "-q"])
    from sentence_transformers import SentenceTransformer

# Visualization
import matplotlib.pyplot as plt

print("All imports successful!")

All imports successful!


In [3]:
# Fix seed for reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

# Device setup
import torch
if torch.cuda.is_available():
    DEVICE = 'cuda'
    print(f"Using GPU: {torch.cuda.get_device_name(0)}")
else:
    DEVICE = 'cpu'
    print("Using CPU")

torch.manual_seed(SEED)
if DEVICE == 'cuda':
    torch.cuda.manual_seed_all(SEED)

Using GPU: Tesla T4


## 2. База знаний и первичный анализ

In [4]:
# Knowledge Base: Machine Learning Concepts
# This is a curated collection of explanations about ML algorithms and concepts

DOCUMENTS = [
    {
        "id": "doc_01",
        "title": "Linear Regression",
        "content": "Linear regression is a fundamental supervised learning algorithm that models the relationship between input features and a continuous target variable. It assumes a linear relationship: y = mx + b. The goal is to find optimal weights that minimize the sum of squared errors between predicted and actual values. Linear regression is fast, interpretable, and works well for problems with linear patterns. However, it may underfit complex nonlinear relationships."
    },
    {
        "id": "doc_02",
        "title": "Logistic Regression",
        "content": "Logistic regression is used for binary classification problems. Despite its name, it is a classification algorithm, not a regression algorithm. It uses the logistic function (sigmoid) to map continuous values to probabilities between 0 and 1. The model outputs a probability that an instance belongs to the positive class. Logistic regression is interpretable, efficient, and provides probability estimates, making it useful for many practical applications."
    },
    {
        "id": "doc_03",
        "title": "Decision Trees",
        "content": "Decision trees are tree-based models that recursively split the feature space into regions. At each node, the algorithm selects the feature that best splits the data (using criteria like Gini impurity or information gain). Decision trees are interpretable and can handle both classification and regression. However, they tend to overfit on training data, especially with deep trees. Ensemble methods like Random Forests and Gradient Boosting address this limitation."
    },
    {
        "id": "doc_04",
        "title": "Random Forests",
        "content": "Random Forests are an ensemble learning method that trains multiple decision trees on random subsets of the data and features. Each tree makes a prediction, and the final prediction is the average (regression) or majority vote (classification) of all trees. Random Forests reduce overfitting compared to individual trees, handle non-linear relationships well, and provide feature importance estimates. They are robust and work well with both numerical and categorical features."
    },
    {
        "id": "doc_05",
        "title": "Support Vector Machines",
        "content": "Support Vector Machines (SVM) find the optimal hyperplane that maximizes the margin between classes in a binary classification problem. SVMs can handle both linear and non-linear classification through the kernel trick, which implicitly maps data to higher dimensions. SVMs are effective in high-dimensional spaces, work well with small to medium-sized datasets, and have solid theoretical foundations. However, they can be slow for large datasets and kernelization adds computational complexity."
    },
    {
        "id": "doc_06",
        "title": "K-Means Clustering",
        "content": "K-Means is an unsupervised learning algorithm for clustering data into k clusters. It iteratively assigns each point to the nearest centroid and updates centroids as the mean of assigned points. K-Means is fast, scalable, and easy to implement. However, the number of clusters k must be specified beforehand, and the algorithm is sensitive to initial centroid placement. K-Means++ and other initialization strategies can improve results and reduce local optima issues."
    },
    {
        "id": "doc_07",
        "title": "Hierarchical Clustering",
        "content": "Hierarchical clustering builds a tree of clusters (dendrogram) by either merging small clusters into larger ones (agglomerative) or splitting large clusters (divisive). Common linkage methods include single, complete, and average linkage. Hierarchical clustering does not require specifying k beforehand and provides a full hierarchy of clusters at different granularities. However, it is computationally expensive for large datasets compared to K-Means."
    },
    {
        "id": "doc_08",
        "title": "Principal Component Analysis",
        "content": "Principal Component Analysis (PCA) is a dimensionality reduction technique that finds the directions (principal components) that explain the most variance in the data. PCA transforms high-dimensional data into a lower-dimensional space while preserving as much variance as possible. It is useful for visualization, noise reduction, and speeding up subsequent algorithms. However, PCA discards low-variance directions, which may contain important patterns, and the resulting components are harder to interpret than original features."
    },
    {
        "id": "doc_09",
        "title": "Gradient Boosting",
        "content": "Gradient Boosting is an ensemble technique that sequentially builds an ensemble of weak learners (usually trees) to correct errors made by previous learners. Each new learner is trained on the residuals of the previous ensemble. Gradient Boosting often achieves state-of-the-art performance on tabular data and handles various loss functions. Popular implementations include XGBoost, LightGBM, and CatBoost. The main drawback is that it can be slow to train and prone to overfitting if not carefully configured."
    },
    {
        "id": "doc_10",
        "title": "Neural Networks",
        "content": "Neural networks are computational models inspired by biological neurons. They consist of interconnected layers of artificial neurons. Each connection has a weight, and neurons apply activation functions to their inputs. Neural networks can learn complex non-linear relationships and are the foundation of deep learning. They require more data than traditional algorithms and are computationally expensive. However, their flexibility and ability to learn hierarchical representations make them powerful for unstructured data like images and text."
    },
    {
        "id": "doc_11",
        "title": "Convolutional Neural Networks",
        "content": "Convolutional Neural Networks (CNNs) are specialized neural networks designed for processing grid-like data such as images. They use convolutional layers that apply learnable filters to detect local features, pooling layers to reduce dimensionality, and fully connected layers for classification. CNNs share weights across space, which reduces the number of parameters. CNNs achieve state-of-the-art results on image recognition, object detection, and other computer vision tasks."
    },
    {
        "id": "doc_12",
        "title": "Recurrent Neural Networks",
        "content": "Recurrent Neural Networks (RNNs) process sequences by maintaining a hidden state that updates as new inputs are processed. RNNs can model temporal dependencies and variable-length sequences. Long Short-Term Memory (LSTM) and Gated Recurrent Unit (GRU) cells address the vanishing gradient problem in standard RNNs. RNNs are effective for time series, natural language processing, and sequence generation. However, they are slower to train than feedforward networks and can be memory-intensive for long sequences."
    },
    {
        "id": "doc_13",
        "title": "Transformers and Attention Mechanisms",
        "content": "Transformers are a neural network architecture based on self-attention mechanisms that allow the model to weigh the importance of different parts of the input. Unlike RNNs, transformers process sequences in parallel, making them faster to train. The attention mechanism computes weighted combinations of values based on query-key similarity. Transformers have become the foundation for state-of-the-art models in NLP (BERT, GPT) and computer vision (Vision Transformer). The self-attention mechanism enables the model to capture long-range dependencies effectively."
    },
    {
        "id": "doc_14",
        "title": "Transfer Learning",
        "content": "Transfer learning involves using a model pre-trained on a large dataset and fine-tuning it for a different but related task. This approach leverages knowledge learned from source task to improve performance on target task. Transfer learning is particularly effective when target task has limited labeled data. Common applications include using ImageNet-trained models for computer vision and pre-trained language models (BERT, GPT) for NLP. Transfer learning reduces training time and improves generalization, especially with small datasets."
    },
    {
        "id": "doc_15",
        "title": "Cross-Validation",
        "content": "Cross-validation is a model evaluation technique that splits data into k folds. The model is trained k times, each time using k-1 folds for training and 1 fold for validation. This provides a more reliable estimate of model performance than a single train-test split. k-fold cross-validation reduces bias and variance in performance estimates. Stratified cross-validation ensures class distribution is maintained in each fold, which is important for imbalanced datasets. Common choices for k are 5 and 10."
    },
    {
        "id": "doc_16",
        "title": "Hyperparameter Tuning",
        "content": "Hyperparameter tuning involves selecting optimal values for parameters that are set before training (e.g., learning rate, regularization strength). Hyperparameters control the learning process and significantly affect model performance. Common tuning strategies include grid search (exhaustive search over parameter space), random search (random sampling), and Bayesian optimization (sequential improvement). Proper hyperparameter tuning often requires cross-validation to ensure results are robust and not overfitted to a specific validation set."
    },
    {
        "id": "doc_17",
        "title": "Regularization Techniques",
        "content": "Regularization techniques add penalties to the model to prevent overfitting. L1 regularization (Lasso) and L2 regularization (Ridge) add penalties proportional to the magnitude of weights. Elastic Net combines L1 and L2 penalties. Dropout randomly deactivates neurons during training to prevent co-adaptation. Early stopping monitors validation performance and stops training when it starts to worsen. Weight decay, batch normalization, and data augmentation are other regularization approaches. Regularization is crucial for building models that generalize well to unseen data."
    },
    {
        "id": "doc_18",
        "title": "Activation Functions",
        "content": "Activation functions introduce non-linearity into neural networks, allowing them to learn complex relationships. Common activation functions include ReLU (Rectified Linear Unit), which is widely used due to its simplicity and effectiveness. Other functions include sigmoid, tanh, leaky ReLU, and ELU. ReLU is preferred in hidden layers because it is computationally efficient and helps mitigate the vanishing gradient problem. Softmax is used in output layers for multi-class classification. The choice of activation function affects training speed and model performance."
    },
    {
        "id": "doc_19",
        "title": "Batch Normalization",
        "content": "Batch normalization normalizes layer inputs by subtracting batch mean and dividing by batch standard deviation. This technique stabilizes training, allows higher learning rates, and reduces internal covariate shift. Batch normalization acts as a regularizer and can reduce the need for careful weight initialization. It is applied to intermediate layers and typically placed after linear transformations and before activation functions. However, batch normalization introduces additional hyperparameters and can behave differently during training and inference."
    },
    {
        "id": "doc_20",
        "title": "Word Embeddings and Word2Vec",
        "content": "Word embeddings represent words as dense vectors in a continuous space where semantically similar words are close together. Word2Vec learns embeddings through two main architectures: Skip-gram (predicting context from word) and CBOW (predicting word from context). These embeddings capture semantic relationships and can be used in downstream NLP tasks. FastText improves upon Word2Vec by learning subword information. Pre-trained embeddings like GloVe and the more recent contextual embeddings from transformers have become standard in NLP."
    }
]

print(f"Total documents: {len(DOCUMENTS)}")
print("\nFirst 3 documents:")
for doc in DOCUMENTS[:3]:
    print(f"\n[{doc['id']}] {doc['title']}")
    print(f"Content: {doc['content'][:150]}...")

Total documents: 20

First 3 documents:

[doc_01] Linear Regression
Content: Linear regression is a fundamental supervised learning algorithm that models the relationship between input features and a continuous target variable....

[doc_02] Logistic Regression
Content: Logistic regression is used for binary classification problems. Despite its name, it is a classification algorithm, not a regression algorithm. It use...

[doc_03] Decision Trees
Content: Decision trees are tree-based models that recursively split the feature space into regions. At each node, the algorithm selects the feature that best ...


### Knowledge Base Analysis

**Domain**: Machine Learning Concepts and Algorithms  
**Why it's suitable for Retrieval**: This KB contains diverse ML concepts (supervised, unsupervised, deep learning) with detailed explanations. Users may ask questions like "What's the difference between linear and logistic regression?", "How do neural networks work?", or "When should I use Random Forests?". The semantic diversity across documents makes it ideal for testing vector-based retrieval.

**Scale**: 20 documents covering fundamental ML concepts.

## 3. Чанкинг документов

In [5]:
def chunk_text(text: str, chunk_size: int = 200, overlap: int = 50) -> List[str]:
    """
    Split text into overlapping chunks.

    Args:
        text: The text to split
        chunk_size: Maximum characters per chunk
        overlap: Number of characters to overlap between chunks

    Returns:
        List of text chunks
    """
    chunks = []
    start = 0

    while start < len(text):
        end = min(start + chunk_size, len(text))
        chunk = text[start:end]
        chunks.append(chunk)

        if end >= len(text):
            break

        start = end - overlap

    return chunks

# Apply chunking to all documents
CHUNK_SIZE = 200
OVERLAP = 50

chunks_with_metadata = []

for doc in DOCUMENTS:
    text_to_chunk = f"{doc['title']}. {doc['content']}"
    chunks = chunk_text(text_to_chunk, chunk_size=CHUNK_SIZE, overlap=OVERLAP)

    for chunk_idx, chunk in enumerate(chunks):
        chunks_with_metadata.append({
            'chunk_id': f"{doc['id']}_chunk_{chunk_idx}",
            'doc_id': doc['id'],
            'doc_title': doc['title'],
            'chunk_text': chunk
        })

print(f"Total chunks after splitting: {len(chunks_with_metadata)}")
print(f"\nExample chunking for document 1 ({DOCUMENTS[0]['title']}):")
doc_1_chunks = [c for c in chunks_with_metadata if c['doc_id'] == DOCUMENTS[0]['id']]
for i, chunk_data in enumerate(doc_1_chunks):
    print(f"\nChunk {i}: {chunk_data['chunk_text'][:100]}...")

Total chunks after splitting: 74

Example chunking for document 1 (Linear Regression):

Chunk 0: Linear Regression. Linear regression is a fundamental supervised learning algorithm that models the ...

Chunk 1: us target variable. It assumes a linear relationship: y = mx + b. The goal is to find optimal weight...

Chunk 2: predicted and actual values. Linear regression is fast, interpretable, and works well for problems w...


### Chunking Parameters

- **chunk_size=200**: Each chunk contains ~200 characters, balancing context and granularity
- **overlap=50**: 50 characters overlap between chunks preserves continuity and captures boundaries better
- **Rationale**: For conceptual documents, a moderately-sized chunk captures enough context to be meaningful while maintaining diversity in the retrieval index

## 4. Эмбеддинги и индекс FAISS

In [6]:
# Load embedding model
print("Loading embedding model (sentence-transformers)...")
embedding_model = SentenceTransformer('all-MiniLM-L6-v2', device=DEVICE)
print(f"Model loaded. Embedding dimension: {embedding_model.get_sentence_embedding_dimension()}")

# Generate embeddings for all chunks
chunk_texts = [c['chunk_text'] for c in chunks_with_metadata]
print(f"\nGenerating embeddings for {len(chunk_texts)} chunks...")

embeddings = embedding_model.encode(chunk_texts, batch_size=32, show_progress_bar=True)
embeddings = embeddings.astype('float32')

print(f"Embeddings shape: {embeddings.shape}")

Loading embedding model (sentence-transformers)...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Model loaded. Embedding dimension: 384

Generating embeddings for 74 chunks...


Batches:   0%|          | 0/3 [00:00<?, ?it/s]

Embeddings shape: (74, 384)


In [7]:
# Build FAISS index
print("Building FAISS index...")
dimension = embeddings.shape[1]
index = faiss.IndexFlatL2(dimension)  # L2 distance (Euclidean)
index.add(embeddings)

print(f"Index built with {index.ntotal} vectors")

Building FAISS index...
Index built with 74 vectors


In [8]:
# Test retrieval with example queries
example_queries = [
    "What is the difference between decision trees and random forests?",
    "How do transformers work?",
    "Clustering algorithms",
    "Neural networks and deep learning",
    "Regularization methods to prevent overfitting"
]

TOP_K = 3

for query in example_queries:
    query_embedding = embedding_model.encode([query], show_progress_bar=False)[0].astype('float32')
    query_embedding = query_embedding.reshape(1, -1)

    distances, indices = index.search(query_embedding, TOP_K)

    print(f"\n{'='*80}")
    print(f"Query: {query}")
    print(f"{'='*80}")

    for rank, idx in enumerate(indices[0]):
        distance = distances[0][rank]
        chunk_meta = chunks_with_metadata[idx]
        print(f"\nRank {rank+1} (distance={distance:.4f}):")
        print(f"  Doc: {chunk_meta['doc_title']}")
        print(f"  Chunk: {chunk_meta['chunk_text'][:100]}...")


Query: What is the difference between decision trees and random forests?

Rank 1 (distance=0.5364):
  Doc: Random Forests
  Chunk: Random Forests. Random Forests are an ensemble learning method that trains multiple decision trees o...

Rank 2 (distance=0.6814):
  Doc: Random Forests
  Chunk: ee makes a prediction, and the final prediction is the average (regression) or majority vote (classi...

Rank 3 (distance=0.9106):
  Doc: Decision Trees
  Chunk: Decision Trees. Decision trees are tree-based models that recursively split the feature space into r...

Query: How do transformers work?

Rank 1 (distance=0.8278):
  Doc: Transformers and Attention Mechanisms
  Chunk: Transformers and Attention Mechanisms. Transformers are a neural network architecture based on self-...

Rank 2 (distance=1.2893):
  Doc: Transformers and Attention Mechanisms
  Chunk:  the importance of different parts of the input. Unlike RNNs, transformers process sequences in para...

Rank 3 (distance=1.3626):
  Doc: Tr

## 5. Контрольные запросы и оценка retrieval

In [9]:
# Define ground truth test queries
# Each query has expected document(s) that should be retrieved

test_queries = [
    {
        "query": "How do linear and logistic regression differ?",
        "expected_docs": ["doc_01", "doc_02"]
    },
    {
        "query": "Explain decision trees and tree-based ensembles",
        "expected_docs": ["doc_03", "doc_04", "doc_09"]
    },
    {
        "query": "What is support vector machine?",
        "expected_docs": ["doc_05"]
    },
    {
        "query": "Clustering methods: K-means and hierarchical",
        "expected_docs": ["doc_06", "doc_07"]
    },
    {
        "query": "Dimensionality reduction techniques",
        "expected_docs": ["doc_08"]
    },
    {
        "query": "Deep learning with neural networks and convolutional networks",
        "expected_docs": ["doc_09", "doc_10", "doc_11"]
    },
    {
        "query": "Recurrent neural networks for sequences",
        "expected_docs": ["doc_12"]
    },
    {
        "query": "Attention & transformers in deep learning",
        "expected_docs": ["doc_13"]
    },
    {
        "query": "How to reuse models with transfer learning",
        "expected_docs": ["doc_14"]
    },
    {
        "query": "Model evaluation and cross-validation",
        "expected_docs": ["doc_15"]
    },
    {
        "query": "Tuning hyperparameters and regularization",
        "expected_docs": ["doc_16", "doc_17"]
    },
    {
        "query": "Word embeddings and semantic representations",
        "expected_docs": ["doc_20"]
    }
]

print(f"Test set: {len(test_queries)} queries")

Test set: 12 queries


In [10]:
def evaluate_retrieval(queries: List[Dict], index: faiss.Index,
                       chunks: List[Dict], model, top_k: int = 5) -> pd.DataFrame:
    """
    Evaluate retrieval quality on test queries.
    """
    results = []

    for test_case in queries:
        query = test_case['query']
        expected_doc_ids = test_case['expected_docs']
        expected_docs = set(expected_doc_ids)
        expected_source = expected_doc_ids[0] if expected_doc_ids else ''

        # Encode query
        query_embedding = model.encode([query], show_progress_bar=False)[0].astype('float32')
        query_embedding = query_embedding.reshape(1, -1)

        # Retrieve
        distances, indices = index.search(query_embedding, top_k)

        retrieved_docs = [chunks[idx]['doc_id'] for idx in indices[0]]
        retrieved_titles = [chunks[idx]['doc_title'] for idx in indices[0]]

        # Calculate metrics
        retrieved_set = set(retrieved_docs)
        hit_at_k = 1 if len(expected_docs & retrieved_set) > 0 else 0

        # Rank of first relevant
        rank_first_relevant = None
        for rank, doc_id in enumerate(retrieved_docs):
            if doc_id in expected_docs:
                rank_first_relevant = rank + 1
                break

        # Recall
        relevant_retrieved = len(expected_docs & retrieved_set)
        recall_at_k = relevant_retrieved / len(expected_docs) if len(expected_docs) > 0 else 0

        results.append({
            'query': query,
            'expected_docs': ', '.join(expected_doc_ids),
            'expected_source': expected_source,
            'retrieved_sources': ' | '.join([f"{t}({d})" for t, d in zip(retrieved_titles, retrieved_docs)]),
            'hit_at_k': hit_at_k,
            'recall_at_k': recall_at_k,
            'rank_of_first_relevant': rank_first_relevant
        })

    return pd.DataFrame(results)

# Evaluate retrieval
eval_df = evaluate_retrieval(test_queries, index, chunks_with_metadata, embedding_model, top_k=5)

print("\nRetrieval Evaluation Results:")
print(eval_df.to_string())

# Calculate aggregate metrics
hit_at_5 = eval_df['hit_at_k'].mean()
recall_at_5 = eval_df['recall_at_k'].mean()
mrr_at_5 = (1 / eval_df['rank_of_first_relevant']).mean()

print(f"\n{'='*60}")
print(f"Aggregate Metrics (k=5):")
print(f"  Hit@5: {hit_at_5:.2%}")
print(f"  Recall@5: {recall_at_5:.2%}")
print(f"  MRR@5: {mrr_at_5:.4f}")
print(f"{'='*60}")


Retrieval Evaluation Results:
                                                            query           expected_docs expected_source                                                                                                                                                                                               retrieved_sources  hit_at_k  recall_at_k  rank_of_first_relevant
0                   How do linear and logistic regression differ?          doc_01, doc_02          doc_01                                                                   Logistic Regression(doc_02) | Linear Regression(doc_01) | Logistic Regression(doc_02) | Linear Regression(doc_01) | Linear Regression(doc_01)         1     1.000000                       1
1                 Explain decision trees and tree-based ensembles  doc_03, doc_04, doc_09          doc_03                                                                                      Decision Trees(doc_03) | Random Forests(doc_04) | Decisi

Save retrieval evaluation results to artifacts

In [11]:
artifacts_dir = "./artifacts"
os.makedirs(artifacts_dir, exist_ok=True)

eval_df.to_csv(os.path.join(artifacts_dir, 'retrieval_eval.csv'), index=False)
print(f"Saved retrieval evaluation to {artifacts_dir}/retrieval_eval.csv")

Saved retrieval evaluation to ./artifacts/retrieval_eval.csv


## 6. Эксперимент: сравнение chunk_size

In [12]:
# Experiment: Compare two chunk_size values
chunk_sizes = [150, 300]
experiment_results = {}

for cs in chunk_sizes:
    print(f"\n{'='*60}")
    print(f"Experiment: chunk_size={cs}")
    print(f"{'='*60}")

    # Re-chunk
    chunks_exp = []
    for doc in DOCUMENTS:
        text = f"{doc['title']}. {doc['content']}"
        chunks = chunk_text(text, chunk_size=cs, overlap=OVERLAP)
        for chunk_idx, chunk in enumerate(chunks):
            chunks_exp.append({
                'chunk_id': f"{doc['id']}_chunk_{chunk_idx}",
                'doc_id': doc['id'],
                'doc_title': doc['title'],
                'chunk_text': chunk
            })

    print(f"Total chunks: {len(chunks_exp)}")

    # Generate embeddings
    chunk_texts_exp = [c['chunk_text'] for c in chunks_exp]
    embeddings_exp = embedding_model.encode(chunk_texts_exp, batch_size=32, show_progress_bar=False)
    embeddings_exp = embeddings_exp.astype('float32')

    # Build index
    index_exp = faiss.IndexFlatL2(embeddings_exp.shape[1])
    index_exp.add(embeddings_exp)

    # Evaluate
    eval_df_exp = evaluate_retrieval(test_queries, index_exp, chunks_exp, embedding_model, top_k=5)

    hit_at_5_exp = eval_df_exp['hit_at_k'].mean()
    recall_at_5_exp = eval_df_exp['recall_at_k'].mean()

    experiment_results[cs] = {
        'chunks': chunks_exp,
        'embeddings': embeddings_exp,
        'index': index_exp,
        'hit_at_5': hit_at_5_exp,
        'recall_at_5': recall_at_5_exp
    }

    print(f"  Hit@5: {hit_at_5_exp:.2%}")
    print(f"  Recall@5: {recall_at_5_exp:.2%}")


Experiment: chunk_size=150
Total chunks: 109
  Hit@5: 100.00%
  Recall@5: 97.22%

Experiment: chunk_size=300
Total chunks: 49
  Hit@5: 100.00%
  Recall@5: 97.22%


In [13]:
# Experiment summary
print(f"\n{'='*70}")
print("EXPERIMENT SUMMARY: chunk_size comparison")
print(f"{'='*70}")
print(f"\n{'chunk_size':<15} {'# chunks':<15} {'Hit@5':<15} {'Recall@5':<15}")
print("-" * 60)

for cs in chunk_sizes:
    num_chunks = len(experiment_results[cs]['chunks'])
    hit = experiment_results[cs]['hit_at_5']
    recall = experiment_results[cs]['recall_at_5']
    print(f"{cs:<15} {num_chunks:<15} {hit:<15.2%} {recall:<15.2%}")

print(f"\n{'='*70}")
print("CONCLUSION:")
if experiment_results[150]['hit_at_5'] >= experiment_results[300]['hit_at_5']:
    print(f"  chunk_size=150 achieved better or equal hit rate.")
    chosen_cs = 150
else:
    print(f"  chunk_size=300 achieved better hit rate.")
    chosen_cs = 300

print(f"  Selected chunk_size={chosen_cs} for main pipeline.")
print(f"{'='*70}")


EXPERIMENT SUMMARY: chunk_size comparison

chunk_size      # chunks        Hit@5           Recall@5       
------------------------------------------------------------
150             109             100.00%         97.22%         
300             49              100.00%         97.22%         

CONCLUSION:
  chunk_size=150 achieved better or equal hit rate.
  Selected chunk_size=150 for main pipeline.


## 7. Обновление базы знаний и переиндексация

In [14]:
# Add new documents to the knowledge base
new_documents = [
    {
        "id": "doc_21",
        "title": "Ensemble Methods and Stacking",
        "content": "Ensemble methods combine multiple models to improve prediction accuracy. Bagging trains models independently on random subsets. Boosting sequentially trains models to correct previous errors. Stacking uses a meta-learner to combine base model predictions. Ensemble methods often outperform individual models by reducing variance and capturing diverse patterns. The key is diversity: ensemble members should make different types of errors."
    },
    {
        "id": "doc_22",
        "title": "Autoencoders and Unsupervised Learning",
        "content": "Autoencoders are neural networks that learn compressed representations by reconstructing their input. They consist of encoder and decoder components. Autoencoders are useful for dimensionality reduction, anomaly detection, and feature learning. Variational Autoencoders (VAE) add a probabilistic interpretation. Denoising autoencoders learn robust representations by reconstructing clean data from noisy input. Autoencoders are versatile unsupervised learning tools applicable to many domains."
    },
    {
        "id": "doc_23",
        "title": "Interpretation and Explainability",
        "content": "Model interpretability is crucial for understanding model decisions and gaining trust. Feature importance methods identify influential input features. SHAP and LIME provide local and global explanations. Attention weights in neural networks highlight important inputs. Saliency maps visualize important regions in images. Adversarial examples reveal model vulnerabilities. Interpretability is especially important in high-stakes domains like healthcare and finance where models must be explainable."
    }
]

print(f"Adding {len(new_documents)} new documents...")
DOCUMENTS_UPDATED = DOCUMENTS + new_documents
print(f"Total documents after update: {len(DOCUMENTS_UPDATED)}")

# Re-chunk with chosen chunk_size
chunks_updated = []
for doc in DOCUMENTS_UPDATED:
    text = f"{doc['title']}. {doc['content']}"
    chunks = chunk_text(text, chunk_size=200, overlap=50)
    for chunk_idx, chunk in enumerate(chunks):
        chunks_updated.append({
            'chunk_id': f"{doc['id']}_chunk_{chunk_idx}",
            'doc_id': doc['id'],
            'doc_title': doc['title'],
            'chunk_text': chunk
        })

print(f"Total chunks after update: {len(chunks_updated)}")

# Generate embeddings for updated KB
print("\nGenerating embeddings for updated KB...")
chunk_texts_updated = [c['chunk_text'] for c in chunks_updated]
embeddings_updated = embedding_model.encode(chunk_texts_updated, batch_size=32, show_progress_bar=False)
embeddings_updated = embeddings_updated.astype('float32')

# Build new index
index_updated = faiss.IndexFlatL2(embeddings_updated.shape[1])
index_updated.add(embeddings_updated)
print(f"New index built with {index_updated.ntotal} vectors")

Adding 3 new documents...
Total documents after update: 23
Total chunks after update: 85

Generating embeddings for updated KB...
New index built with 85 vectors


In [15]:
# Compare retrieval before and after update
print("Comparing retrieval quality before and after KB update...")
print(f"\n{'='*80}")

before_after_results = []

select_test_queries = test_queries[:6]

for test_case in select_test_queries:
    query = test_case['query']

    query_emb = embedding_model.encode([query], show_progress_bar=False)[0].astype('float32')
    query_emb = query_emb.reshape(1, -1)

    _, indices_before = index.search(query_emb, 5)
    retrieved_before = ' | '.join([chunks_with_metadata[idx]['doc_title'] for idx in indices_before[0]])

    _, indices_after = index_updated.search(query_emb, 5)
    retrieved_after = ' | '.join([chunks_updated[idx]['doc_title'] for idx in indices_after[0]])

    changed = "Yes" if retrieved_before != retrieved_after else "No"

    before_after_results.append({
        'query': query,
        'before_retrieved_sources': retrieved_before,
        'after_retrieved_sources': retrieved_after,
        'changed': changed
    })

    print(f"Query: {query}")
    print(f"  Before: {retrieved_before}")
    print(f"  After:  {retrieved_after}")
    print(f"  Changed: {changed}\n")

before_after_df = pd.DataFrame(before_after_results)
before_after_df.to_csv(os.path.join(artifacts_dir, 'retrieval_before_after_update.csv'), index=False)
print(f"Saved before/after comparison to artifacts/retrieval_before_after_update.csv")

Comparing retrieval quality before and after KB update...

Query: How do linear and logistic regression differ?
  Before: Logistic Regression | Linear Regression | Logistic Regression | Linear Regression | Linear Regression
  After:  Logistic Regression | Linear Regression | Logistic Regression | Linear Regression | Linear Regression
  Changed: No

Query: Explain decision trees and tree-based ensembles
  Before: Decision Trees | Random Forests | Decision Trees | Random Forests | Decision Trees
  After:  Decision Trees | Random Forests | Decision Trees | Random Forests | Ensemble Methods and Stacking
  Changed: Yes

Query: What is support vector machine?
  Before: Support Vector Machines | Support Vector Machines | Gradient Boosting | Cross-Validation | Decision Trees
  After:  Support Vector Machines | Support Vector Machines | Gradient Boosting | Cross-Validation | Decision Trees
  Changed: No

Query: Clustering methods: K-means and hierarchical
  Before: K-Means Clustering | Hierarch

## 8. Mini-RAG

In [16]:
# Simple template-based LLM replacement (no actual LLM, just intelligent templates)
class SimpleRAG:
    def __init__(self, index, chunks, embedding_model, top_k=3):
        self.index = index
        self.chunks = chunks
        self.embedding_model = embedding_model
        self.top_k = top_k

    def retrieve(self, query: str) -> Tuple[List[Dict], float]:
        """Retrieve top-k relevant chunks."""
        query_emb = self.embedding_model.encode([query], show_progress_bar=False)[0].astype('float32')
        query_emb = query_emb.reshape(1, -1)

        distances, indices = self.index.search(query_emb, self.top_k)

        retrieved = []
        for idx in indices[0]:
            retrieved.append({
                'title': self.chunks[idx]['doc_title'],
                'text': self.chunks[idx]['chunk_text'],
                'doc_id': self.chunks[idx]['doc_id']
            })

        relevance_score = 1.0 - (distances[0][0] / 100)  # Normalize distance to score
        return retrieved, relevance_score

    def generate_answer(self, query: str, retrieved_docs: List[Dict]) -> str:
        """Generate answer based on retrieved documents."""
        context = "\n".join([f"From '{doc['title']}': {doc['text']}" for doc in retrieved_docs])

        # Simple template-based generation
        answer = f"""
Based on the knowledge base:

{context}

Summary: The relevant documents discuss {retrieved_docs[0]['title']},
which provides information about your query on '{query[:50]}...'.
For more details, refer to the source documents listed below.
        """.strip()

        return answer

    def answer(self, query: str) -> Dict:
        """Generate complete RAG answer with sources."""
        retrieved_docs, relevance = self.retrieve(query)
        answer_text = self.generate_answer(query, retrieved_docs)

        sources = '; '.join([f"{doc['title']} ({doc['doc_id']})" for doc in retrieved_docs])

        return {
            'question': query,
            'answer': answer_text,
            'retrieved_sources': sources,
            'relevance_score': relevance
        }

# Initialize mini-RAG
rag = SimpleRAG(index_updated, chunks_updated, embedding_model, top_k=3)

In [17]:
# Test mini-RAG on example questions
rag_questions = [
    "What is gradient boosting and how does it work?",
    "Explain the difference between clustering and classification.",
    "How are autoencoders used in machine learning?",
    "What are the advantages of transfer learning?",
    "Describe the role of attention mechanisms in transformers.",
]

rag_results = []

for q in rag_questions:
    result = rag.answer(q)
    rag_results.append({
        'question': result['question'],
        'answer': result['answer'],
        'retrieved_sources': result['retrieved_sources'],
        'relevance_score': f"{result['relevance_score']:.3f}"
    })

    print(f"\n{'='*80}")
    print(f"Q: {result['question']}")
    print(f"\nA: {result['answer']}")
    print(f"\nSources: {result['retrieved_sources']}")
    print(f"Relevance score: {result['relevance_score']}")

rag_df = pd.DataFrame(rag_results)
rag_df.to_csv(os.path.join(artifacts_dir, 'rag_examples.csv'), index=False)
print(f"\nSaved RAG examples to artifacts/rag_examples.csv")


Q: What is gradient boosting and how does it work?

A: Based on the knowledge base:

From 'Gradient Boosting': Gradient Boosting. Gradient Boosting is an ensemble technique that sequentially builds an ensemble of weak learners (usually trees) to correct errors made by previous learners. Each new learner is tra
From 'Gradient Boosting': made by previous learners. Each new learner is trained on the residuals of the previous ensemble. Gradient Boosting often achieves state-of-the-art performance on tabular data and handles various loss
From 'Ensemble Methods and Stacking': subsets. Boosting sequentially trains models to correct previous errors. Stacking uses a meta-learner to combine base model predictions. Ensemble methods often outperform individual models by reducing

Summary: The relevant documents discuss Gradient Boosting,
which provides information about your query on 'What is gradient boosting and how does it work?...'.
For more details, refer to the source documents listed below

## 9. Анализ ошибок mini-RAG

In [18]:
# Test some problematic queries where retrieval might fail
problematic_queries = [
    "What is blockchain?",  # Not in KB
    "GPU vs CPU for machine learning",  # Mentioned but not primary focus
    "How to deploy a model to production?",  # Not covered
    "Explain SHAP values",  # Briefly mentioned in interpretability doc
]

print("\nANALYSIS OF PROBLEMATIC CASES:")
print(f"{'='*80}")

for i, q in enumerate(problematic_queries, 1):
    result = rag.answer(q)
    print(f"\n{i}. Query: {q}")
    print(f"   Retrieved: {result['retrieved_sources']}")
    print(f"   Relevance: {result['relevance_score']:.3f}")
    print(f"   Issue: ", end="")

    if result['relevance_score'] < 0.3:
        print("Low relevance - query not well covered in KB")
    elif "blockchain" in q.lower() or "deploy" in q.lower():
        print("Topic not in knowledge base")
    elif result['relevance_score'] < 0.5:
        print("Partial relevance - related concepts retrieved")
    else:
        print("Retrieved relevant documents")


ANALYSIS OF PROBLEMATIC CASES:

1. Query: What is blockchain?
   Retrieved: Logistic Regression (doc_02); Linear Regression (doc_01); Neural Networks (doc_10)
   Relevance: 0.986
   Issue: Topic not in knowledge base

2. Query: GPU vs CPU for machine learning
   Retrieved: Neural Networks (doc_10); Support Vector Machines (doc_05); Regularization Techniques (doc_17)
   Relevance: 0.987
   Issue: Retrieved relevant documents

3. Query: How to deploy a model to production?
   Retrieved: Ensemble Methods and Stacking (doc_21); Transfer Learning (doc_14); Ensemble Methods and Stacking (doc_21)
   Relevance: 0.983
   Issue: Topic not in knowledge base

4. Query: Explain SHAP values
   Retrieved: Interpretation and Explainability (doc_23); Interpretation and Explainability (doc_23); Logistic Regression (doc_02)
   Relevance: 0.987
   Issue: Retrieved relevant documents


## 10. Итоговый анализ

In [19]:
# Summary statistics
print("\n" + "="*80)
print("SUMMARY STATISTICS")
print("="*80)

print(f"\nKnowledge Base:")
print(f"  - Total documents (initial): {len(DOCUMENTS)}")
print(f"  - Total documents (updated): {len(DOCUMENTS_UPDATED)}")
print(f"  - Total chunks (updated): {len(chunks_updated)}")
print(f"  - Embedding dimension: {embeddings_updated.shape[1]}")

print(f"\nRetrieval Performance (initial index, k=5):")
print(f"  - Hit@5: {eval_df['hit_at_k'].mean():.2%}")
print(f"  - Recall@5: {eval_df['recall_at_k'].mean():.2%}")
print(f"  - MRR@5: {(1 / eval_df['rank_of_first_relevant']).mean():.4f}")

print(f"\nChunk Size Experiment:")
for cs in chunk_sizes:
    print(f"  - chunk_size={cs}: {len(experiment_results[cs]['chunks'])} chunks, "
          f"Hit@5={experiment_results[cs]['hit_at_5']:.2%}")

print(f"\nBefore/After Update:")
changed_count = (before_after_df['changed'] == 'Yes').sum()
print(f"  - Queries with changed results: {changed_count}/{len(before_after_df)}")

print(f"\nMini-RAG Evaluation:")
print(f"  - Test questions: {len(rag_questions)}")
print(f"  - Average relevance score: {rag_df['relevance_score'].astype(float).mean():.3f}")

print(f"\nArtifacts saved:")
print(f"  - retrieval_eval.csv")
print(f"  - rag_examples.csv")
print(f"  - retrieval_before_after_update.csv")

print("\n" + "="*80)


SUMMARY STATISTICS

Knowledge Base:
  - Total documents (initial): 20
  - Total documents (updated): 23
  - Total chunks (updated): 85
  - Embedding dimension: 384

Retrieval Performance (initial index, k=5):
  - Hit@5: 100.00%
  - Recall@5: 94.44%
  - MRR@5: 0.9583

Chunk Size Experiment:
  - chunk_size=150: 109 chunks, Hit@5=100.00%
  - chunk_size=300: 49 chunks, Hit@5=100.00%

Before/After Update:
  - Queries with changed results: 2/6

Mini-RAG Evaluation:
  - Test questions: 5
  - Average relevance score: 0.994

Artifacts saved:
  - retrieval_eval.csv
  - rag_examples.csv
  - retrieval_before_after_update.csv

